In [82]:
from langgraph.graph import StateGraph , START , END
from langchain_openrouter import ChatOpenRouter
from pydantic import BaseModel , Field
from typing import TypedDict , Annotated , Literal
from dotenv import load_dotenv
load_dotenv()

True

In [72]:
model = ChatOpenRouter(
    model = "openrouter/free"
)

In [73]:
class sentiment_structure(BaseModel):
    sentiment: Annotated[ 
        Literal["positive" , "negative"] ,
        Field(
            description = "sentiment can either positive or negative"
        )
    ]
structured_sentiment_model = model.with_structured_output(sentiment_structure)

In [74]:
class ReviewState(TypedDict):
    review:str
    sentiment:str
    diagnosis_neg_review:str
    negative_response:str
    positive_response:str

In [75]:
def review_sentiment(state:ReviewState):
    rev = state["review"]

    prompt = f"you're given a customer review your work is to tell me what's the sentiment of customer in single word either in positive or negative all cases should be in lowercase. Review :\n{rev}"

    response = structured_sentiment_model.invoke(prompt)

    return {
        "sentiment" : response.sentiment
    }

def response_to_give(state:ReviewState) -> Literal["positive_sentiment_response" , "diagnos_problem"]:
    senti = state["sentiment"]

    if senti == "positive":
        return "positive_sentiment_response"
    else:
        return "diagnos_problem"

def positive_sentiment_response(state:ReviewState):
    rev = state["review"]
    prompt = f"so it is already analyzed that sentiment of this review is positive, your task it to give a short positive response. NOTE->don't decorate response, give it in simple paragraph"

    response = model.invoke(prompt)

    return {
        "positive_response" : response.content
    }

def diagnos_problem(state:ReviewState):
    rev = state["review"]
    prompt = f"it is already analyzed that sentiment of this review is negative , your task is to give a short diagnosis for customer problem in bullet points. NOTE->don't decorate response, give it in simple paragraph"
    response = model.invoke(prompt)
    return {
        "diagnosis_neg_review":response.content
    }

def negative_sentiment_response(state:ReviewState):
    rev = state["review"]
    diagnos = state["diagnosis_neg_review"]

    prompt = f"it's already analyzed that customet review is negative and the diagnosis of that problem is already given you task is to give a good response for customer inconvinence on the bases of his negative review . Review:\n{rev}\nDiagnosis:\n{diagnos}\nNOTE->\n1. don't throw up in the air give practical response , be practical what actually company do\n2.don't decorate response, give it in simple paragraph"

    response = model.invoke(prompt)

    return {
        "negative_response" : response.content
    }

In [76]:
graph = StateGraph(ReviewState)

# Nodes
graph.add_node("review_sentiment" , review_sentiment)
graph.add_node("positive_sentiment_response" , positive_sentiment_response)
graph.add_node("diagnos_problem" , diagnos_problem)
graph.add_node("negative_sentiment_response" , negative_sentiment_response)

# Edges
graph.add_edge(START , "review_sentiment")
graph.add_conditional_edges("review_sentiment" , response_to_give)
graph.add_edge("diagnos_problem" , "negative_sentiment_response")
graph.add_edge("negative_sentiment_response" , END)
graph.add_edge("positive_sentiment_response" , END)

# compile
workflow = graph.compile()

In [77]:

pos_initial_state = {
    "review" : "Absolutely loved the product! The quality is excellent, delivery was fast, and the overall experience was amazing."
}
neg_initial_state = {
    "review" : "I am very disappointed with the product. The quality is poor, it stopped working after a few days, and the customer support was not helpful at all."
}

In [78]:
pos_final_state = workflow.invoke(pos_initial_state)

In [79]:
pos_final_state

{'review': 'Absolutely loved the product! The quality is excellent, delivery was fast, and the overall experience was amazing.',
 'sentiment': 'positive',
 'positive_response': "Thank you so much for your kind words! We're glad you enjoyed your experience and truly appreciate your positive feedback. It means a lot to us, and we look forward to serving you again."}

In [80]:
neg_final_state = workflow.invoke(neg_initial_state)

In [81]:
neg_final_state

{'review': 'I am very disappointed with the product. The quality is poor, it stopped working after a few days, and the customer support was not helpful at all.',
 'sentiment': 'negative',
 'diagnosis_neg_review': 'User Safety: safe',
 'negative_response': "I am very sorry to hear that your product stopped working after a few days and that your previous experience with our customer support was unhelpful. We expect our products to last much longer and our support team to provide actual solutions rather than frustration. To make this right, please reply to this message with your order number so we can immediately process a full refund or send a replacement, whichever you prefer. We are also reviewing your previous support interaction internally to ensure this doesn't happen again, and we will get this resolved for you as quickly as possible."}